# Verificación de entorno — GenAI Multimodal Course

**Abre este notebook primero.** Si las celdas corren sin error, estás listo para el curso.

---

## Paso 1: Crear tu API key en Google AI Studio

1. Ve a **[aistudio.google.com](https://aistudio.google.com)**
2. Inicia sesión con tu cuenta de Google
3. Panel izquierdo → **"Get API key"** → **"Create API key"** → "Create API key in new project"
4. Copia la key (empieza con `AIza...`)

**Importante:** No vinculen una *Billing account* para que sigan en el Free Tier.
---

## Paso 2: Guardar la API key según tu entorno

### Opción A — Ejecución local (con `.env`)

Crea un archivo `.env` en la raíz del repo con este contenido:

```
GOOGLE_API_KEY=AIza...
```

> El archivo `.env` ya está en `.gitignore` — nunca se sube al repo.

### Opción B — Google Colab (Secrets)

1. Haz clic en el ícono 🔑 en el panel izquierdo de Colab
2. Nombre: `GOOGLE_API_KEY` · Valor: tu key
3. Activa el toggle **"Notebook access"**

---

## Paso 3: Instalar dependencias y ejecutar las celdas de validación


In [1]:
# Instalar dependencias del curso
# python-dotenv permite leer el .env en ejecución local
!pip install google-genai chromadb numpy Pillow python-dotenv --quiet


[notice] A new release of pip available: 22.2.2 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# Verificar versiones instaladas
import google.genai
import chromadb
import numpy
import PIL

print(f"google-genai : {google.genai.__version__}")
print(f"chromadb     : {chromadb.__version__}")
print(f"numpy        : {numpy.__version__}")
print(f"Pillow       : {PIL.__version__}")
print()
print("OK — dependencias instaladas.")

google-genai : 1.74.0
chromadb     : 1.5.8
numpy        : 2.2.6
Pillow       : 12.2.0

OK — dependencias instaladas.


In [3]:
# Cargar API key — detecta automáticamente Colab o entorno local con .env
import os
from google import genai

def load_api_key() -> str:
    # Opción A: entorno local con archivo .env
    try:
        from dotenv import load_dotenv
        load_dotenv()  # carga variables de .env al entorno
        key = os.environ.get("GOOGLE_API_KEY")
        if key:
            print("API key cargada desde .env")
            return key
    except ImportError:
        pass

    # Opción B: Google Colab Secrets
    try:
        from google.colab import userdata
        key = userdata.get("GOOGLE_API_KEY")
        if key:
            print("API key cargada desde Colab Secrets")
            return key
    except Exception:
        pass

    raise EnvironmentError(
        "No se encontró GOOGLE_API_KEY.\n"
        "Local: crea un archivo .env con GOOGLE_API_KEY=AIza...\n"
        "Colab: agrega la key en el panel de Secrets (ícono 🔑)."
    )


GOOGLE_API_KEY = load_api_key()
client = genai.Client(api_key=GOOGLE_API_KEY)
print("Cliente Gemini inicializado.")

API key cargada desde .env
Cliente Gemini inicializado.


In [4]:
# Test 1: generación de texto
MODEL = "gemini-3.1-flash-lite-preview"

response = client.models.generate_content(
    model=MODEL,
    contents="Di 'entorno verificado' en español, nada más."
)
print(f"Gemini dice: {response.text}")
print("OK — generate_content funciona.")

Gemini dice: Entorno verificado
OK — generate_content funciona.


In [11]:
# Test 2: embeddings
EMBED_MODEL = "gemini-embedding-2"

result = client.models.embed_content(
    model=EMBED_MODEL,
    contents="transferencia internacional Yape BCP"
)
embedding = result.embeddings[0].values
print(f"Embedding dim: {len(embedding)}")  # debe ser 768
print(f"Primeros 5 valores: {embedding[:5]}")
print("OK — embeddings funcionan.")

Embedding dim: 3072
Primeros 5 valores: [-0.004018073, 0.0038195578, 0.006805833, 0.01523566, 0.036283057]
OK — embeddings funcionan.


---

Si las celdas corrieron sin error: **estás listo**. Abre `session1_genai_sdk_fundamentals.ipynb`.

**Problemas comunes:**
- `EnvironmentError: No se encontró GOOGLE_API_KEY`:
  - Local: verifica que el archivo `.env` existe en la raíz del repo y tiene `GOOGLE_API_KEY=AIza...`
  - Colab: verifica que activaste el toggle "Notebook access" en Secrets
- `401 API_KEY_INVALID`: la key no está bien copiada.
- `429 RESOURCE_EXHAUSTED`: límite gratuito. Espera 1 minuto o usa otra cuenta Google.
- `ModuleNotFoundError`: ejecuta la celda de `pip install` de nuevo y reinicia el kernel.